In [3]:
"""
策略：移动止损 双向（根据市场状态过滤 + 30m 价格区间过滤）
1. 根据市场状态过滤（STRONG_UP 只做多，STRONG_DOWN 只做空）
2. 15m 趋势判断（多头：HH + HL + EMA 多头排列；空头：LL + LH + EMA 空头排列）
3. 1m 入场条件：
   - 多头：价格回落到 EMA20 且成交量放大，且不处于最近 30m 区间的上 25%（避免追高）
   - 空头：价格反弹到 EMA20 且成交量放大，且不处于最近 30m 区间的下 25%（避免追空）
4. 移动止损：
   - 多头：用最近 HL 的 low 计算止损位
   - 空头：用最近 HH 的 high 计算止损位
"""
import pandas as pd
import numpy as np

# ======================
# 参数区
# ======================
FEE_RATE = 0.0005       # 0.05% taker
LEVERAGE = 20

ATR_MULTIPLIER = 10      # ATR 乘数，用于止损计算
HL_LOOKBACK = 20         # 用多少根 1m K 线判断 HL (或 HH)，默认15
HL_BUFFER = 1            # 用于多头移动止损
HH_BUFFER = 1            # 用于空头移动止损

# 30m 价格区间过滤阈值
ENTRY_ZONE_RATIO = 0.03  # 价格进入最近 30m 区间的上/下 25% 时，视为“已经过度偏离”，先不入场

# ======================
# 技术指标
# ======================
def EMA(series, period):
    return series.ewm(span=period, adjust=False).mean()


def ATR(df, period=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ======================
# 加载 1m 数据
# ======================
df_1m = pd.read_csv("RIVERUSDT_data_1m_25_10_12.csv", parse_dates=['timestamp'])
df_1m.set_index('timestamp', inplace=True)

# ======================
# 构建 15m 数据
# ======================
df_15m = df_1m.resample('15T').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}).dropna()

# ======================
# 15m 技术因子
# ======================
df_15m['EMA20'] = EMA(df_15m['close'], 20)
df_15m['EMA50'] = EMA(df_15m['close'], 50)
df_15m['EMA200'] = EMA(df_15m['close'], 200)

# 结构因子
N = 1
df_15m['HH'] = df_15m['high'] > df_15m['high'].rolling(N).max().shift()
df_15m['HL'] = df_15m['low']  > df_15m['low'].rolling(N).min().shift()
df_15m['LL'] = df_15m['low']  < df_15m['low'].rolling(N).min().shift()
df_15m['LH'] = df_15m['high'] < df_15m['high'].rolling(N).max().shift()

df_15m['ALLOW_LONG'] = (
    df_15m['HH'] &
    df_15m['HL'] &
    (df_15m['EMA20'] > df_15m['EMA50']) &
    (df_15m['EMA50'] > df_15m['EMA200'])
)

df_15m['ALLOW_SHORT'] = (
    df_15m['LL'] &
    df_15m['LH'] &
    (df_15m['EMA20'] < df_15m['EMA50']) &
    (df_15m['EMA50'] < df_15m['EMA200'])
)

# ======================
# REGIME（市场状态）
# ======================
df_15m['EMA20_slope'] = (
    df_15m['EMA20'] - df_15m['EMA20'].shift(5)
) / df_15m['close']

def market_regime(row):
    if (
        row['close'] > row['EMA200'] and
        row['EMA20'] > row['EMA50'] > row['EMA200'] and
        row['EMA20_slope'] > 0.0001
    ):
        return 'STRONG_UP'

    elif (
        row['close'] < row['EMA200'] and
        row['EMA20'] < row['EMA50'] < row['EMA200'] and
        row['EMA20_slope'] < -0.0001
    ):
        return 'STRONG_DOWN'

    elif abs(row['EMA20_slope']) < 0.00005:
        return 'RANGE'

    else:
        return 'WEAK'

df_15m['REGIME'] = df_15m.apply(market_regime, axis=1)

# ======================
# 对齐到 1m
# ======================
df_1m['ALLOW_LONG'] = df_15m['ALLOW_LONG'].reindex(df_1m.index, method='ffill')
df_1m['ALLOW_SHORT'] = df_15m['ALLOW_SHORT'].reindex(df_1m.index, method='ffill')
df_1m['REGIME'] = df_15m['REGIME'].reindex(df_1m.index, method='ffill')

# ======================
# 1m 技术因子
# ======================
df_1m['EMA20'] = EMA(df_1m['close'], 20)
df_1m['ATR'] = ATR(df_1m)

df_1m['VOL_SPIKE'] = (
    df_1m['volume'] >
    1.15 * df_1m['volume'].rolling(20).mean()
)

# ======================
# 根据高低结构因子判定,如N=20,则判定60分钟的极值
# ======================
df_1m['LOW_EXTREME'] = df_1m['low'].rolling(N*3).min().shift(1)
df_1m['HIGH_EXTREME'] = df_1m['high'].rolling(N*3).max().shift(1)
df_1m['RANGE_EXTREME'] = df_1m['HIGH_EXTREME'] - df_1m['LOW_EXTREME']

# 经典做法：避免在最近 30m 区间的上 25% / 下 25% 处追单
# 也就是价格已经明显冲到最近区间上端或下端时，先不入场
# 这样更像“回踩进场”，而不是“追高/追空”
df_1m['LONG_ENTRY_LIMIT'] = df_1m['HIGH_EXTREME'] * (1 - ENTRY_ZONE_RATIO)
df_1m['SHORT_ENTRY_LIMIT'] = df_1m['LOW_EXTREME'] * (1 + ENTRY_ZONE_RATIO)

# ======================
# 回测主循环
# ======================
trades = []
position = None

for t in range(50, len(df_1m)):
    row = df_1m.iloc[t]

    # ===== 开仓 =====
    if position is None:

        # 强多市场只做多
        long_ok = (
            row['REGIME'] == 'STRONG_UP' and
            row['ALLOW_LONG'] and
            row['low'] <= row['EMA20'] and
            row['VOL_SPIKE'] and
            row['close'] <= row['LONG_ENTRY_LIMIT']
        )

        # 强空市场只做空
        short_ok = (
            row['REGIME'] == 'STRONG_DOWN' and
            row['ALLOW_SHORT'] and
            row['high'] >= row['EMA20'] and
            row['VOL_SPIKE'] and
            row['close'] >= row['SHORT_ENTRY_LIMIT']
        )

        if long_ok:
            entry = row['close']
            stop = entry - ATR_MULTIPLIER * row['ATR']
            position = {
                'side': 'long',
                'entry_price': entry,
                'stop': stop,
                'entry_time': df_1m.index[t]
            }

        elif short_ok:
            entry = row['close']
            stop = entry + ATR_MULTIPLIER * row['ATR']
            position = {
                'side': 'short',
                'entry_price': entry,
                'stop': stop,
                'entry_time': df_1m.index[t]
            }

    # ===== 持仓管理 =====
    else:
        low, high = row['low'], row['high']

        if position['side'] == 'long':
            recent_lows = df_1m['low'].iloc[t-HL_LOOKBACK:t]
            hl_stop = recent_lows.min() * HL_BUFFER
            position['stop'] = max(position['stop'], hl_stop)

            if low <= position['stop']:
                exit_price = position['stop']
                ret = (exit_price - position['entry_price']) / position['entry_price']
                net = LEVERAGE * ret - 2 * FEE_RATE * LEVERAGE
                trades.append({
                    'side': 'long',
                    'entry_time': position['entry_time'],
                    'exit_time': df_1m.index[t],
                    'return': net
                })
                position = None

        else:
            recent_highs = df_1m['high'].iloc[t-HL_LOOKBACK:t]
            hh_stop = recent_highs.max() * HH_BUFFER
            position['stop'] = min(position['stop'], hh_stop)

            if high >= position['stop']:
                exit_price = position['stop']
                ret = (position['entry_price'] - exit_price) / position['entry_price']
                net = LEVERAGE * ret - 2 * FEE_RATE * LEVERAGE
                trades.append({
                    'side': 'short',
                    'entry_time': position['entry_time'],
                    'exit_time': df_1m.index[t],
                    'return': net
                })
                position = None

# ======================
# 结果统计
# ======================
df_trades = pd.DataFrame(trades)
print("15分钟数据预览:")
print(df_15m.head())

# print("1分钟数据预览：")
# print(df_1m.head())

# print("交易明细：")
# print(df_trades.head(20))
df_trades.to_csv("trades.csv", index=False)
print("交易次数:", len(df_trades))
print("胜率:", (df_trades['return'] > 0).mean())

# 单利计算模式
equity = df_trades['return'].cumsum()
equity.to_csv("equity.csv", index=False)
drawdown = equity.cummax() - equity

# 复利计算模式
compound_equity = (1 + df_trades['return']).cumprod() - 1
compound_equity.to_csv("compound_equity.csv", index=False)
compound_drawdown = compound_equity.cummax() - compound_equity

print("累计收益(单利):", equity.iloc[-1] if len(df_trades) else 0)
print("累计收益(复利):", compound_equity.iloc[-1] if len(df_trades) else 0)
print("最大回撤(单利):", drawdown.max())
print("最大回撤(复利):", compound_drawdown.max())

print("\n按方向统计：")
print(df_trades.groupby('side')['return'].agg(['count', 'mean', 'sum']))


/var/folders/h_/2rpkgm1n1y5gpfss71sfwmgr0000gn/T/ipykernel_13273/209187007.py:52: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_15m = df_1m.resample('15T').agg({


15分钟数据预览:
                      open   high    low  close     volume     EMA20  \
timestamp                                                              
2025-10-17 14:45:00  4.500  4.671  3.616  3.875  2433270.2  3.875000   
2025-10-17 15:00:00  3.883  3.928  3.366  3.392  1182324.0  3.829000   
2025-10-17 15:15:00  3.391  3.547  3.129  3.265  1386418.9  3.775286   
2025-10-17 15:30:00  3.270  3.955  3.226  3.526  1807672.4  3.751544   
2025-10-17 15:45:00  3.526  3.619  3.281  3.351   917416.7  3.713397   

                        EMA50    EMA200     HH     HL     LL     LH  \
timestamp                                                             
2025-10-17 14:45:00  3.875000  3.875000  False  False  False  False   
2025-10-17 15:00:00  3.856059  3.870194  False  False   True   True   
2025-10-17 15:15:00  3.832880  3.864172  False  False   True   True   
2025-10-17 15:30:00  3.820846  3.860807   True   True  False  False   
2025-10-17 15:45:00  3.802420  3.855735  False   True  Fals